In [0]:
%sql
CREATE OR REPLACE TABLE payments.default.feature_register (
  feature_id            STRING    COMMENT 'Feature key (F001...)',
  feature_name          STRING    COMMENT 'Column name in the target table',
  business_name         STRING    COMMENT 'Business-friendly name',
  description           STRING    COMMENT 'Business definition',
  data_domain           STRING    COMMENT 'Business domain the feature belongs to',
  source_table          STRING    COMMENT 'Raw table the feature is derived from',
  source_columns        STRING    COMMENT 'Raw columns used (comma separated)',
  depends_on            STRING    COMMENT 'Other derived features it depends on',
  target_table          STRING    COMMENT 'Table where the feature lives',
  granularity           STRING    COMMENT 'Level of one row: User or Transaction',
  script_name           STRING    COMMENT 'Script that creates the feature',
  data_type             STRING    COMMENT 'Output data type',
  feature_category      STRING    COMMENT 'Cleaning / Ratio / Banding / Aggregation / Segmentation / Risk',
  sql_expression        STRING    COMMENT 'SQL logic used to compute the feature',
  null_handling         STRING    COMMENT 'What happens with NULL / zero inputs',
  allowed_values        STRING    COMMENT 'Expected domain or range',
  data_owner            STRING    COMMENT 'Accountable business owner',
  data_steward          STRING    COMMENT 'Day-to-day manager of the definition and quality',
  technical_custodian   STRING    COMMENT 'Team operating the pipeline',
  sensitivity           STRING    COMMENT 'Public / Internal / Confidential / Restricted',
  personal_data         STRING    COMMENT 'Personal data under GDPR: Yes / No',
  regulatory_notes      STRING    COMMENT 'Regulatory or ethical usage notes',
  critical_data_element STRING    COMMENT 'Critical data element: Yes / No',
  data_quality_rule     STRING    COMMENT 'Data quality expectation in plain words',
  dq_check_ids          STRING    COMMENT 'Checks in dq_check_register that test this feature',
  refresh_frequency     STRING    COMMENT 'How often the feature is recomputed',
  retention             STRING    COMMENT 'Retention period',
  status                STRING    COMMENT 'Active / Under review / Deprecated',
  version               STRING    COMMENT 'Feature definition version',
  assumptions           STRING    COMMENT 'Assumptions made when building the feature',
  known_issues          STRING    COMMENT 'Open issues raised during review',
  approved_by           STRING    COMMENT 'Who approved the definition',
  approval_date         DATE      COMMENT 'When it was approved',
  last_reviewed_by      STRING    COMMENT 'Who did the last periodic review',
  review_date           DATE      COMMENT 'Date of the last periodic review',
  next_review_date      DATE      COMMENT 'review_date + 12 months',
  review_notes          STRING    COMMENT 'Outcome of the last review',
  registered_by         STRING    COMMENT 'User who ran the script',
  registered_at         TIMESTAMP COMMENT 'When the register was (re)built'
)
COMMENT 'Feature register for pay_users_features and pay_transactions_features: definitions, ownership, classification, quality and lineage';
 
INSERT INTO payments.default.feature_register
SELECT
  feature_id,
  feature_name,
  business_name,
  description,
  data_domain,
  source_table,
  source_columns,
  depends_on,
  target_table,
  granularity,
  script_name,
  data_type,
  feature_category,
  sql_expression,
  null_handling,
  allowed_values,
  data_owner,
  data_steward,
  technical_custodian,
  sensitivity,
  personal_data,
  regulatory_notes,
  critical_data_element,
  data_quality_rule,
  dq_check_ids,
  refresh_frequency,
  retention,
  status,
  version,
  assumptions,
  known_issues,
  CAST(NULL AS STRING) AS approved_by,
  CAST(NULL AS DATE) AS approval_date,
  CAST(NULL AS STRING) AS last_reviewed_by,
  CAST(NULL AS DATE) AS review_date,
  CAST(NULL AS DATE) AS next_review_date,
  CAST(NULL AS STRING) AS review_notes,
  current_user() AS registered_by,
  current_timestamp() AS registered_at
FROM VALUES
 
-- ---------------- USERS ----------------
("F001", "yearly_income", "Annual income", "Annual income converted from currency text (e.g. $45,000) to a number.",
 "Customer / Credit", "payments.default.pay_users", "yearly_income", "-",
 "payments.default.pay_users_features", "User", "feature_engineering.sql", "DOUBLE",
 "Cleaning", "TRY_CAST(REGEXP_REPLACE(CAST(yearly_income AS STRING), '[$,]', '') AS DOUBLE)", "Unparseable values become NULL", ">= 0",
 "TO_DEFINE (suggested: Head of Credit Risk)", "TO_DEFINE", "Data Engineering - Databricks (payments.default)", "Confidential",
 "Yes", "Personal financial data linked to an individual", "Yes", "Parse rate >= 99% (NULL only where raw value is NULL); value >= 0",
 "DQ-006, DQ-007", "On each run of feature_engineering.sql (manual)", "TO_DEFINE - align with data retention policy", "Active",
 "1.0", "Values are USD", "Keeps the raw column name; original text value is not retained"),
("F002", "total_debt", "Total debt", "Total debt converted from currency text to a number.",
 "Customer / Credit", "payments.default.pay_users", "total_debt", "-",
 "payments.default.pay_users_features", "User", "feature_engineering.sql", "DOUBLE",
 "Cleaning", "TRY_CAST(REGEXP_REPLACE(CAST(total_debt AS STRING), '[$,]', '') AS DOUBLE)", "Unparseable values become NULL", ">= 0",
 "TO_DEFINE (suggested: Head of Credit Risk)", "TO_DEFINE", "Data Engineering - Databricks (payments.default)", "Confidential",
 "Yes", "Personal financial data linked to an individual", "Yes", "Parse rate >= 99% (NULL only where raw value is NULL); value >= 0",
 "DQ-008, DQ-009", "On each run of feature_engineering.sql (manual)", "TO_DEFINE - align with data retention policy", "Active",
 "1.0", "Values are USD", "Keeps the raw column name; original text value is not retained"),
("F003", "per_capita_income", "Area per-capita income", "Per-capita income of the user's area converted from currency text to a number.",
 "Customer / Credit", "payments.default.pay_users", "per_capita_income", "-",
 "payments.default.pay_users_features", "User", "feature_engineering.sql", "DOUBLE",
 "Cleaning", "TRY_CAST(REGEXP_REPLACE(CAST(per_capita_income AS STRING), '[$,]', '') AS DOUBLE)", "Unparseable values become NULL", ">= 0",
 "TO_DEFINE (suggested: Head of Credit Risk)", "TO_DEFINE", "Data Engineering - Databricks (payments.default)", "Internal",
 "No", "-", "No", "Parse rate >= 99% (NULL only where raw value is NULL); value >= 0",
 "DQ-010, DQ-011", "On each run of feature_engineering.sql (manual)", "TO_DEFINE - align with data retention policy", "Active",
 "1.0", "Values are USD", "Keeps the raw column name; original text value is not retained"),
("F004", "debt_to_income", "Debt-to-income ratio", "Total debt divided by yearly income.",
 "Customer / Credit", "payments.default.pay_users", "total_debt, yearly_income", "total_debt, yearly_income",
 "payments.default.pay_users_features", "User", "feature_engineering.sql", "DOUBLE",
 "Ratio", "ROUND(total_debt / NULLIF(yearly_income, 0), 2)", "NULL when income is 0 or NULL", ">= 0",
 "TO_DEFINE (suggested: Head of Credit Risk)", "TO_DEFINE", "Data Engineering - Databricks (payments.default)", "Confidential",
 "Yes", "Personal financial data linked to an individual", "Yes", "NULL only when denominator is 0 or NULL; review values above the 99th percentile",
 "DQ-014, DQ-015, DQ-016", "On each run of feature_engineering.sql (manual)", "TO_DEFINE - align with data retention policy", "Active",
 "1.0", "-", "Very large values when income is close to 0"),
("F005", "income_vs_area", "Income vs area average", "User income relative to the per-capita income of their area (1 = area average).",
 "Customer / Credit", "payments.default.pay_users", "yearly_income, per_capita_income", "yearly_income, per_capita_income",
 "payments.default.pay_users_features", "User", "feature_engineering.sql", "DOUBLE",
 "Ratio", "ROUND(yearly_income / NULLIF(per_capita_income, 0), 2)", "NULL when per-capita income is 0 or NULL", ">= 0",
 "TO_DEFINE (suggested: Head of Credit Risk)", "TO_DEFINE", "Data Engineering - Databricks (payments.default)", "Confidential",
 "Yes", "Personal financial data linked to an individual", "No", "NULL only when denominator is 0 or NULL; review values above the 99th percentile",
 "DQ-017, DQ-018", "On each run of feature_engineering.sql (manual)", "TO_DEFINE - align with data retention policy", "Active",
 "1.0", "-", "-"),
("F006", "years_to_retirement", "Years to retirement", "Years left until retirement age.",
 "Customer / Credit", "payments.default.pay_users", "retirement_age, current_age", "-",
 "payments.default.pay_users_features", "User", "feature_engineering.sql", "INT",
 "Ratio", "GREATEST(retirement_age - current_age, 0)", "NULL if either age is NULL", ">= 0",
 "TO_DEFINE (suggested: Head of Credit Risk)", "TO_DEFINE", "Data Engineering - Databricks (payments.default)", "Confidential",
 "Yes", "Age-derived: do not use as a decision driver for credit (non-discrimination)", "No", "NULL only when denominator is 0 or NULL; review values above the 99th percentile",
 "DQ-019, DQ-020", "On each run of feature_engineering.sql (manual)", "TO_DEFINE - align with data retention policy", "Active",
 "1.0", "-", "Clamped at 0: the original version was negative for retirees"),
("F007", "is_retired", "Retired flag", "TRUE when the user has reached retirement age.",
 "Customer / Credit", "payments.default.pay_users", "current_age, retirement_age", "-",
 "payments.default.pay_users_features", "User", "feature_engineering.sql", "BOOLEAN",
 "Banding", "current_age >= retirement_age", "NULL if either age is NULL", "TRUE / FALSE / NULL",
 "TO_DEFINE (suggested: Head of Credit Risk)", "TO_DEFINE", "Data Engineering - Databricks (payments.default)", "Confidential",
 "Yes", "Age-derived: do not use as a decision driver for credit (non-discrimination)", "No", "Values only from the allowed list; Unknown / Non valutabile share < 5%",
 "DQ-021", "On each run of feature_engineering.sql (manual)", "TO_DEFINE - align with data retention policy", "Active",
 "1.0", "-", "-"),
("F008", "debt_per_card", "Debt per credit card", "Average debt per credit card.",
 "Customer / Credit", "payments.default.pay_users", "total_debt, num_credit_cards", "total_debt",
 "payments.default.pay_users_features", "User", "feature_engineering.sql", "DOUBLE",
 "Ratio", "ROUND(total_debt / NULLIF(num_credit_cards, 0), 0)", "NULL when the user has 0 cards", ">= 0",
 "TO_DEFINE (suggested: Head of Credit Risk)", "TO_DEFINE", "Data Engineering - Databricks (payments.default)", "Confidential",
 "Yes", "Personal financial data linked to an individual", "No", "NULL only when denominator is 0 or NULL; review values above the 99th percentile",
 "DQ-022", "On each run of feature_engineering.sql (manual)", "TO_DEFINE - align with data retention policy", "Active",
 "1.0", "-", "Users with debt but 0 cards get NULL, not flagged"),
("F009", "credit_band", "Credit score band", "FICO-style credit score band.",
 "Customer / Credit", "payments.default.pay_users", "credit_score", "-",
 "payments.default.pay_users_features", "User", "feature_engineering.sql", "STRING",
 "Banding", "CASE WHEN credit_score IS NULL THEN 'Unknown' WHEN credit_score < 580 THEN 'Poor' WHEN credit_score < 670 THEN 'Fair' WHEN credit_score < 740 THEN 'Good' WHEN credit_score < 800 THEN 'Very Good' ELSE 'Excellent' END", "NULL score -> Unknown", "Poor, Fair, Good, Very Good, Excellent, Unknown",
 "TO_DEFINE (suggested: Head of Credit Risk)", "TO_DEFINE", "Data Engineering - Databricks (payments.default)", "Confidential",
 "Yes", "Creditworthiness profiling: GDPR Art. 22 applies if used for automated decisions; needs human review and explainability", "No", "Values only from the allowed list; Unknown / Non valutabile share < 5%",
 "DQ-023, DQ-024, DQ-025", "On each run of feature_engineering.sql (manual)", "TO_DEFINE - align with data retention policy", "Active",
 "1.0", "Standard FICO band cut-offs", "Fixed: NULL score was previously labelled Excellent"),
("F010", "generation", "Generation", "Generation based on birth year.",
 "Customer / Credit", "payments.default.pay_users", "birth_year", "-",
 "payments.default.pay_users_features", "User", "feature_engineering.sql", "STRING",
 "Banding", "CASE WHEN birth_year IS NULL THEN 'Unknown' WHEN birth_year >= 1997 THEN 'Gen Z' WHEN birth_year >= 1981 THEN 'Millennial' WHEN birth_year >= 1965 THEN 'Gen X' WHEN birth_year >= 1946 THEN 'Boomer' ELSE 'Silent' END", "NULL year -> Unknown", "Gen Z, Millennial, Gen X, Boomer, Silent, Unknown",
 "TO_DEFINE (suggested: Head of Credit Risk)", "TO_DEFINE", "Data Engineering - Databricks (payments.default)", "Confidential",
 "Yes", "Age-derived: do not use as a decision driver for credit (non-discrimination)", "No", "Values only from the allowed list; Unknown / Non valutabile share < 5%",
 "DQ-026, DQ-027, DQ-028", "On each run of feature_engineering.sql (manual)", "TO_DEFINE - align with data retention policy", "Active",
 "1.0", "Pew Research generation cut-offs", "Fixed: NULL and pre-1946 years were previously labelled Boomer"),
("F011", "credit_risk_level", "Credit risk level", "Credit risk tier combining credit score and debt-to-income.",
 "Customer / Credit", "payments.default.pay_users", "credit_score, total_debt, yearly_income", "debt_to_income",
 "payments.default.pay_users_features", "User", "feature_engineering.sql", "STRING",
 "Risk", "CASE WHEN credit_score IS NULL AND debt_to_income IS NULL THEN 'Non valutabile' WHEN credit_score < 620 OR debt_to_income > 1.5 THEN 'Alto' WHEN credit_score < 700 OR debt_to_income > 0.8 THEN 'Medio' ELSE 'Basso' END", "Both inputs NULL -> Non valutabile", "Alto, Medio, Basso, Non valutabile",
 "TO_DEFINE (suggested: Head of Credit Risk)", "TO_DEFINE", "Data Engineering - Databricks (payments.default)", "Confidential",
 "Yes", "Creditworthiness profiling: GDPR Art. 22 applies if used for automated decisions; needs human review and explainability", "Yes", "Values only from the allowed list; Unknown / Non valutabile share < 5%",
 "DQ-029, DQ-030, DQ-031", "On each run of feature_engineering.sql (manual)", "TO_DEFINE - align with data retention policy", "Under review",
 "1.0", "Medio rule (score < 700 OR DTI > 0.8) reconstructed from a truncated cell", "Score thresholds (620/700) not aligned with credit_band cut-offs (580/670/740)"),
("F012", "fraud_risk_level", "Over-indebtedness risk level (named fraud)", "Rule-based risk tier on debt level, number of cards and income vs area.",
 "Customer / Credit", "payments.default.pay_users", "total_debt, yearly_income, per_capita_income, num_credit_cards", "yearly_income, total_debt, per_capita_income",
 "payments.default.pay_users_features", "User", "feature_engineering.sql", "STRING",
 "Risk", "CASE WHEN yearly_income IS NULL OR total_debt IS NULL THEN 'Non valutabile' WHEN total_debt > yearly_income * 3 OR (num_credit_cards >= 6 AND yearly_income < per_capita_income) THEN 'Critical' WHEN total_debt > yearly_income * 2 OR yearly_income > per_capita_income * 5 THEN 'Suspicious' ELSE 'Normal' END", "NULL income or debt -> Non valutabile", "Critical, Suspicious, Normal, Non valutabile",
 "TO_DEFINE (suggested: Head of Credit Risk)", "TO_DEFINE", "Data Engineering - Databricks (payments.default)", "Confidential",
 "Yes", "Creditworthiness profiling: GDPR Art. 22 applies if used for automated decisions; needs human review and explainability", "Yes", "Values only from the allowed list; Unknown / Non valutabile share < 5%",
 "DQ-032, DQ-033, DQ-034", "On each run of feature_engineering.sql (manual)", "TO_DEFINE - align with data retention policy", "Under review",
 "1.0", "-", "Measures over-indebtedness, not fraud; flags high earners (> 5x area income) as Suspicious; overlaps with credit_risk_level. Rename or redefine."),
("F013", "flag_many_cards_low_income", "Many cards + low income flag", "TRUE when the user has 6+ credit cards and earns below the area per-capita income.",
 "Customer / Credit", "payments.default.pay_users", "num_credit_cards, yearly_income, per_capita_income", "yearly_income, per_capita_income",
 "payments.default.pay_users_features", "User", "feature_engineering.sql", "BOOLEAN",
 "Risk", "CASE WHEN num_credit_cards IS NULL OR yearly_income IS NULL OR per_capita_income IS NULL THEN NULL ELSE num_credit_cards >= 6 AND yearly_income < per_capita_income END", "Any NULL input -> NULL", "TRUE / FALSE / NULL",
 "TO_DEFINE (suggested: Head of Credit Risk)", "TO_DEFINE", "Data Engineering - Databricks (payments.default)", "Confidential",
 "Yes", "Creditworthiness profiling: GDPR Art. 22 applies if used for automated decisions; needs human review and explainability", "No", "Values only from the allowed list; Unknown / Non valutabile share < 5%",
 "DQ-035", "On each run of feature_engineering.sql (manual)", "TO_DEFINE - align with data retention policy", "Under review",
 "1.0", "Definition copied from the Critical rule of fraud_risk_level; original definition not available", "-"),
 
-- ---------------- TRANSACTIONS ----------------
("F014", "amount", "Transaction amount", "Transaction amount converted from currency text (e.g. $-77.00) to a number.",
 "Payments", "payments.default.pay_transactions", "amount", "-",
 "payments.default.pay_transactions_features", "Transaction", "feature_engineering.sql", "DOUBLE",
 "Cleaning", "TRY_CAST(REGEXP_REPLACE(CAST(amount AS STRING), '[$,]', '') AS DOUBLE)", "Unparseable values become NULL", "Any; negative = refund",
 "TO_DEFINE (suggested: Head of Payments)", "TO_DEFINE", "Data Engineering - Databricks (payments.default)", "Confidential",
 "Yes", "Personal financial data linked to an individual", "Yes", "Parse rate >= 99% (NULL only where raw value is NULL)",
 "DQ-012, DQ-013", "On each run of feature_engineering.sql (manual)", "TO_DEFINE - align with data retention policy", "Active",
 "1.0", "Values are USD; negative amounts are refunds", "Keeps the raw column name; original text value is not retained"),
("F015", "city_amount_level", "Merchant city spend tier", "Tier of the merchant city by total transacted amount (tertiles of cities).",
 "Payments", "payments.default.pay_transactions", "merchant_city, merchant_state, amount", "amount",
 "payments.default.pay_transactions_features", "Transaction", "feature_engineering.sql", "STRING",
 "Segmentation", "SUM(amount) GROUP BY merchant_city, merchant_state (excl. ONLINE, empty, NULL); CASE WHEN CUME_DIST() OVER (ORDER BY city_total_amount) <= 1/3 THEN 'Low Level' WHEN ... <= 2/3 THEN 'Middle Level' ELSE 'High Level' END; ONLINE -> 'Online'; no match -> 'Città non disponibile'", "Missing city -> Città non disponibile", "Low Level, Middle Level, High Level, Online, Città non disponibile",
 "TO_DEFINE (suggested: Head of Payments)", "TO_DEFINE", "Data Engineering - Databricks (payments.default)", "Internal",
 "No", "-", "No", "Values only from the allowed list; Unknown / Non valutabile share < 5%",
 "DQ-036, DQ-037, DQ-038", "On each run of feature_engineering.sql (manual)", "TO_DEFINE - align with data retention policy", "Active",
 "1.0", "Tertiles are by number of cities, not by share of amount", "Totals are net of refunds; labels in English while others are Italian"),
("F016", "customer_n_transactions", "Customer transaction count", "Total number of transactions of the client (repeated on each of their rows).",
 "Payments", "payments.default.pay_transactions", "client_id", "-",
 "payments.default.pay_transactions_features", "Transaction", "feature_engineering.sql", "BIGINT",
 "Aggregation", "COUNT(*) GROUP BY client_id (amount NOT NULL)", "NULL client_id -> NULL", ">= 1",
 "TO_DEFINE (suggested: Head of Payments)", "TO_DEFINE", "Data Engineering - Databricks (payments.default)", "Confidential",
 "Yes", "Personal financial data linked to an individual", "No", "Reconciles to source: per-client figures match the cleaned transactions",
 "DQ-039, DQ-040", "On each run of feature_engineering.sql (manual)", "TO_DEFINE - align with data retention policy", "Active",
 "1.0", "Counts all history, no time window", "Refund rows are counted as transactions"),
("F017", "customer_total_amount", "Customer total spend", "Total amount spent by the client (repeated on each of their rows).",
 "Payments", "payments.default.pay_transactions", "client_id, amount", "amount",
 "payments.default.pay_transactions_features", "Transaction", "feature_engineering.sql", "DOUBLE",
 "Aggregation", "SUM(amount) GROUP BY client_id", "NULL client_id -> NULL", "Any",
 "TO_DEFINE (suggested: Head of Payments)", "TO_DEFINE", "Data Engineering - Databricks (payments.default)", "Confidential",
 "Yes", "Personal financial data linked to an individual", "Yes", "Reconciles to source: per-client figures match the cleaned transactions",
 "DQ-041", "On each run of feature_engineering.sql (manual)", "TO_DEFINE - align with data retention policy", "Active",
 "1.0", "Counts all history, no time window", "Net of refunds"),
("F018", "customer_frequency_level", "Customer frequency tier", "Client tier by number of transactions (tertiles of clients).",
 "Payments", "payments.default.pay_transactions", "client_id", "customer_n_transactions",
 "payments.default.pay_transactions_features", "Transaction", "feature_engineering.sql", "STRING",
 "Segmentation", "CASE WHEN CUME_DIST() OVER (ORDER BY customer_n_transactions) <= 1/3 THEN 'Bassa frequenza' WHEN ... <= 2/3 THEN 'Media frequenza' ELSE 'Alta frequenza' END", "No client -> Cliente non disponibile", "Bassa frequenza, Media frequenza, Alta frequenza, Cliente non disponibile",
 "TO_DEFINE (suggested: Head of Payments)", "TO_DEFINE", "Data Engineering - Databricks (payments.default)", "Confidential",
 "Yes", "Customer profiling: GDPR Art. 21 right to object if used for marketing", "No", "Values only from the allowed list; Unknown / Non valutabile share < 5%",
 "DQ-042, DQ-043, DQ-044", "On each run of feature_engineering.sql (manual)", "TO_DEFINE - align with data retention policy", "Active",
 "1.0", "CUME_DIST used instead of NTILE so ties share a tier", "-"),
("F019", "customer_monetary_level", "Customer spend tier", "Client tier by total amount spent (tertiles of clients).",
 "Payments", "payments.default.pay_transactions", "client_id, amount", "customer_total_amount",
 "payments.default.pay_transactions_features", "Transaction", "feature_engineering.sql", "STRING",
 "Segmentation", "CASE WHEN CUME_DIST() OVER (ORDER BY customer_total_amount) <= 1/3 THEN 'Bassa spesa' WHEN ... <= 2/3 THEN 'Media spesa' ELSE 'Alta spesa' END", "No client -> Cliente non disponibile", "Bassa spesa, Media spesa, Alta spesa, Cliente non disponibile",
 "TO_DEFINE (suggested: Head of Payments)", "TO_DEFINE", "Data Engineering - Databricks (payments.default)", "Confidential",
 "Yes", "Customer profiling: GDPR Art. 21 right to object if used for marketing", "No", "Values only from the allowed list; Unknown / Non valutabile share < 5%",
 "DQ-045, DQ-046, DQ-047", "On each run of feature_engineering.sql (manual)", "TO_DEFINE - align with data retention policy", "Under review",
 "1.0", "Labels reconstructed from a truncated cell", "Net of refunds")
AS v(feature_id, feature_name, business_name, description, data_domain, source_table, source_columns, depends_on, target_table, granularity, script_name, data_type, feature_category, sql_expression, null_handling, allowed_values, data_owner, data_steward, technical_custodian, sensitivity, personal_data, regulatory_notes, critical_data_element, data_quality_rule, dq_check_ids, refresh_frequency, retention, status, version, assumptions, known_issues);
 
 


In [0]:
%sql
-- =====================================================================
-- TABLE 4: DATA QUALITY CHECK REGISTER
-- Every check_sql returns failed_rows and total_rows.
-- A check passes when failed_rows / total_rows * 100 <= max_failed_pct.
-- =====================================================================
CREATE OR REPLACE TABLE payments.default.dq_check_register (
  check_id          STRING    COMMENT 'Check key (DQ-001...)',
  feature_id        STRING    COMMENT 'Feature tested (feature_register.feature_id) or TABLE for table-level checks',
  feature_name      STRING    COMMENT 'Feature tested',
  target_table      STRING    COMMENT 'Table the check runs on',
  dq_dimension      STRING    COMMENT 'Completeness / Validity / Accuracy / Consistency / Uniqueness / Timeliness',
  check_description STRING    COMMENT 'What the check verifies',
  check_sql         STRING    COMMENT 'Query returning failed_rows, total_rows',
  max_failed_pct    DOUBLE    COMMENT 'Tolerance: max % of failed rows for a PASS',
  severity          STRING    COMMENT 'Blocker = stop BI refresh; Warning = publish and investigate',
  frequency         STRING    COMMENT 'When the check runs',
  check_owner       STRING    COMMENT 'Who follows up on failures',
  status            STRING    COMMENT 'Active / Disabled',
  registered_at     TIMESTAMP COMMENT 'When the register was (re)built'
)
COMMENT 'Data quality checks for the features in feature_register';
 
INSERT INTO payments.default.dq_check_register
SELECT check_id, feature_id, feature_name, target_table, dq_dimension, check_description, check_sql,
       CAST(max_failed_pct AS DOUBLE), severity, frequency, check_owner, status, current_timestamp()
FROM VALUES
("DQ-001", "TABLE", "(table level)", "payments.default.pay_users_features", "Consistency",
 "Row count of pay_users_features equals the raw pay_users table",
 "SELECT ABS((SELECT COUNT(*) FROM payments.default.pay_users_features) - (SELECT COUNT(*) FROM payments.default.pay_users)) AS failed_rows, (SELECT COUNT(*) FROM payments.default.pay_users) AS total_rows",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-002", "TABLE", "(table level)", "payments.default.pay_users_features", "Uniqueness",
 "User id is unique",
 "SELECT COUNT(*) - COUNT(DISTINCT id) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-003", "TABLE", "(table level)", "payments.default.pay_transactions_features", "Consistency",
 "Row count of pay_transactions_features equals the raw pay_transactions table (joins must not duplicate rows)",
 "SELECT ABS((SELECT COUNT(*) FROM payments.default.pay_transactions_features) - (SELECT COUNT(*) FROM payments.default.pay_transactions)) AS failed_rows, (SELECT COUNT(*) FROM payments.default.pay_transactions) AS total_rows",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-004", "TABLE", "(table level)", "payments.default.pay_transactions_features", "Uniqueness",
 "Transaction id is unique",
 "SELECT COUNT(*) - COUNT(DISTINCT id) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_transactions_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-005", "TABLE", "(table level)", "payments.default.pay_transactions_features", "Consistency",
 "Every client_id exists in pay_users_features (referential integrity)",
 "SELECT COUNT_IF(u.id IS NULL) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_transactions_features t LEFT JOIN payments.default.pay_users_features u ON t.client_id = u.id WHERE t.client_id IS NOT NULL",
 0, "Warning", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-006", "F001", "yearly_income", "payments.default.pay_users_features", "Completeness",
 "yearly_income is populated (parse failures show up as NULL)",
 "SELECT COUNT_IF(yearly_income IS NULL) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 1, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-007", "F001", "yearly_income", "payments.default.pay_users_features", "Validity",
 "yearly_income is not negative",
 "SELECT COUNT_IF(yearly_income < 0) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-008", "F002", "total_debt", "payments.default.pay_users_features", "Completeness",
 "total_debt is populated (parse failures show up as NULL)",
 "SELECT COUNT_IF(total_debt IS NULL) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 1, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-009", "F002", "total_debt", "payments.default.pay_users_features", "Validity",
 "total_debt is not negative",
 "SELECT COUNT_IF(total_debt < 0) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-010", "F003", "per_capita_income", "payments.default.pay_users_features", "Completeness",
 "per_capita_income is populated (parse failures show up as NULL)",
 "SELECT COUNT_IF(per_capita_income IS NULL) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 1, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-011", "F003", "per_capita_income", "payments.default.pay_users_features", "Validity",
 "per_capita_income is not negative",
 "SELECT COUNT_IF(per_capita_income < 0) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-012", "F014", "amount", "payments.default.pay_transactions_features", "Completeness",
 "amount is populated (parse failures show up as NULL)",
 "SELECT COUNT_IF(amount IS NULL) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_transactions_features",
 0.1, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-013", "F014", "amount", "payments.default.pay_transactions_features", "Accuracy",
 "Total cleaned amount matches the raw source after parsing (difference < 0.01)",
 "SELECT CASE WHEN ABS(a - b) > 0.01 THEN 1 ELSE 0 END AS failed_rows, 1 AS total_rows FROM (SELECT (SELECT SUM(amount) FROM payments.default.pay_transactions_features) AS a, (SELECT SUM(TRY_CAST(REGEXP_REPLACE(CAST(amount AS STRING), '[$,]', '') AS DOUBLE)) FROM payments.default.pay_transactions) AS b)",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-014", "F004", "debt_to_income", "payments.default.pay_users_features", "Consistency",
 "debt_to_income is NULL only when income is 0/NULL or debt is NULL",
 "SELECT COUNT_IF(debt_to_income IS NULL AND yearly_income > 0 AND total_debt IS NOT NULL) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-015", "F004", "debt_to_income", "payments.default.pay_users_features", "Validity",
 "debt_to_income is not negative",
 "SELECT COUNT_IF(debt_to_income < 0) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-016", "F004", "debt_to_income", "payments.default.pay_users_features", "Accuracy",
 "Outliers: debt_to_income above 10 (likely income data error)",
 "SELECT COUNT_IF(debt_to_income > 10) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 1, "Warning", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-017", "F005", "income_vs_area", "payments.default.pay_users_features", "Consistency",
 "income_vs_area is NULL only when an input is missing or per-capita income is 0",
 "SELECT COUNT_IF(income_vs_area IS NULL AND per_capita_income > 0 AND yearly_income IS NOT NULL) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-018", "F005", "income_vs_area", "payments.default.pay_users_features", "Validity",
 "income_vs_area is not negative",
 "SELECT COUNT_IF(income_vs_area < 0) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-019", "F006", "years_to_retirement", "payments.default.pay_users_features", "Validity",
 "years_to_retirement is not negative",
 "SELECT COUNT_IF(years_to_retirement < 0) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-020", "F006", "years_to_retirement", "payments.default.pay_users_features", "Consistency",
 "Retired users have 0 years to retirement",
 "SELECT COUNT_IF(is_retired AND years_to_retirement <> 0) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-021", "F007", "is_retired", "payments.default.pay_users_features", "Completeness",
 "is_retired is populated",
 "SELECT COUNT_IF(is_retired IS NULL) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 1, "Warning", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-022", "F008", "debt_per_card", "payments.default.pay_users_features", "Consistency",
 "debt_per_card is NULL only when the user has 0 cards or debt is missing",
 "SELECT COUNT_IF(debt_per_card IS NULL AND num_credit_cards > 0 AND total_debt IS NOT NULL) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-023", "F009", "credit_band", "payments.default.pay_users_features", "Validity",
 "credit_band only takes allowed values",
 "SELECT COUNT_IF(credit_band IS NULL OR credit_band NOT IN ('Poor', 'Fair', 'Good', 'Very Good', 'Excellent', 'Unknown')) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-024", "F009", "credit_band", "payments.default.pay_users_features", "Completeness",
 "Share of Unknown credit_band is below 5%",
 "SELECT COUNT_IF(credit_band = 'Unknown') AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 5, "Warning", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-025", "F009", "credit_band", "payments.default.pay_users_features", "Consistency",
 "Excellent band only for credit_score >= 800 (regression test for the NULL bug)",
 "SELECT COUNT_IF(credit_band = 'Excellent' AND (credit_score IS NULL OR credit_score < 800)) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-026", "F010", "generation", "payments.default.pay_users_features", "Validity",
 "generation only takes allowed values",
 "SELECT COUNT_IF(generation IS NULL OR generation NOT IN ('Gen Z', 'Millennial', 'Gen X', 'Boomer', 'Silent', 'Unknown')) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-027", "F010", "generation", "payments.default.pay_users_features", "Completeness",
 "Share of Unknown generation is below 5%",
 "SELECT COUNT_IF(generation = 'Unknown') AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 5, "Warning", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-028", "F010", "generation", "payments.default.pay_users_features", "Consistency",
 "Boomer only for birth years 1946-1964 (regression test)",
 "SELECT COUNT_IF(generation = 'Boomer' AND (birth_year IS NULL OR birth_year < 1946 OR birth_year > 1964)) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-029", "F011", "credit_risk_level", "payments.default.pay_users_features", "Validity",
 "credit_risk_level only takes allowed values",
 "SELECT COUNT_IF(credit_risk_level IS NULL OR credit_risk_level NOT IN ('Alto', 'Medio', 'Basso', 'Non valutabile')) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-030", "F011", "credit_risk_level", "payments.default.pay_users_features", "Completeness",
 "Share of Non valutabile is below 5%",
 "SELECT COUNT_IF(credit_risk_level = 'Non valutabile') AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 5, "Warning", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-031", "F011", "credit_risk_level", "payments.default.pay_users_features", "Consistency",
 "Users with credit_score < 620 are always Alto",
 "SELECT COUNT_IF(credit_score < 620 AND credit_risk_level <> 'Alto') AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-032", "F012", "fraud_risk_level", "payments.default.pay_users_features", "Validity",
 "fraud_risk_level only takes allowed values",
 "SELECT COUNT_IF(fraud_risk_level IS NULL OR fraud_risk_level NOT IN ('Critical', 'Suspicious', 'Normal', 'Non valutabile')) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-033", "F012", "fraud_risk_level", "payments.default.pay_users_features", "Completeness",
 "Share of Non valutabile is below 5%",
 "SELECT COUNT_IF(fraud_risk_level = 'Non valutabile') AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 5, "Warning", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-034", "F012", "fraud_risk_level", "payments.default.pay_users_features", "Accuracy",
 "Plausibility: Critical share stays below 10% of users",
 "SELECT COUNT_IF(fraud_risk_level = 'Critical') AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 10, "Warning", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-035", "F013", "flag_many_cards_low_income", "payments.default.pay_users_features", "Consistency",
 "Flagged users are always Critical in fraud_risk_level",
 "SELECT COUNT_IF(flag_many_cards_low_income AND fraud_risk_level <> 'Critical') AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_users_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-036", "F015", "city_amount_level", "payments.default.pay_transactions_features", "Validity",
 "city_amount_level only takes allowed values",
 "SELECT COUNT_IF(city_amount_level IS NULL OR city_amount_level NOT IN ('Low Level', 'Middle Level', 'High Level', 'Online', 'Città non disponibile')) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_transactions_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-037", "F015", "city_amount_level", "payments.default.pay_transactions_features", "Completeness",
 "Share of Città non disponibile is below 2%",
 "SELECT COUNT_IF(city_amount_level = 'Città non disponibile') AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_transactions_features",
 2, "Warning", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-038", "F015", "city_amount_level", "payments.default.pay_transactions_features", "Consistency",
 "Each city + state has exactly one tier",
 "SELECT COUNT_IF(n_levels > 1) AS failed_rows, COUNT(*) AS total_rows FROM (SELECT merchant_city, merchant_state, COUNT(DISTINCT city_amount_level) AS n_levels FROM payments.default.pay_transactions_features WHERE merchant_city IS NOT NULL GROUP BY merchant_city, merchant_state)",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-039", "F016", "customer_n_transactions", "payments.default.pay_transactions_features", "Accuracy",
 "customer_n_transactions equals the actual number of the client's rows with an amount",
 "SELECT COUNT_IF(n_rows <> n_feature) AS failed_rows, COUNT(*) AS total_rows FROM (SELECT client_id, COUNT(*) AS n_rows, MAX(customer_n_transactions) AS n_feature FROM payments.default.pay_transactions_features WHERE client_id IS NOT NULL AND amount IS NOT NULL GROUP BY client_id)",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-040", "F016", "customer_n_transactions", "payments.default.pay_transactions_features", "Validity",
 "customer_n_transactions is at least 1 when present",
 "SELECT COUNT_IF(customer_n_transactions < 1) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_transactions_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-041", "F017", "customer_total_amount", "payments.default.pay_transactions_features", "Accuracy",
 "customer_total_amount equals the sum of the client's amounts (difference < 0.01)",
 "SELECT COUNT_IF(ABS(s - f) > 0.01) AS failed_rows, COUNT(*) AS total_rows FROM (SELECT client_id, SUM(amount) AS s, MAX(customer_total_amount) AS f FROM payments.default.pay_transactions_features WHERE client_id IS NOT NULL AND amount IS NOT NULL GROUP BY client_id)",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-042", "F018", "customer_frequency_level", "payments.default.pay_transactions_features", "Validity",
 "customer_frequency_level only takes allowed values",
 "SELECT COUNT_IF(customer_frequency_level IS NULL OR customer_frequency_level NOT IN ('Bassa frequenza', 'Media frequenza', 'Alta frequenza', 'Cliente non disponibile')) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_transactions_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-043", "F018", "customer_frequency_level", "payments.default.pay_transactions_features", "Consistency",
 "Each client has exactly one tier",
 "SELECT COUNT_IF(n_levels > 1) AS failed_rows, COUNT(*) AS total_rows FROM (SELECT client_id, COUNT(DISTINCT customer_frequency_level) AS n_levels FROM payments.default.pay_transactions_features WHERE client_id IS NOT NULL GROUP BY client_id)",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-044", "F018", "customer_frequency_level", "payments.default.pay_transactions_features", "Accuracy",
 "Tier balance: each tier holds 20-45% of clients (tertiles)",
 "SELECT COUNT_IF((lvl = 'Bassa frequenza' AND (share < 0.20 OR share > 0.45)) OR (lvl = 'Media frequenza' AND (share < 0.20 OR share > 0.45)) OR (lvl = 'Alta frequenza' AND (share < 0.20 OR share > 0.45))) AS failed_rows, COUNT(*) AS total_rows FROM (SELECT customer_frequency_level AS lvl, COUNT(DISTINCT client_id) / SUM(COUNT(DISTINCT client_id)) OVER () AS share FROM payments.default.pay_transactions_features WHERE client_id IS NOT NULL AND customer_frequency_level IN ('Bassa frequenza', 'Media frequenza', 'Alta frequenza') GROUP BY customer_frequency_level)",
 0, "Warning", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-045", "F019", "customer_monetary_level", "payments.default.pay_transactions_features", "Validity",
 "customer_monetary_level only takes allowed values",
 "SELECT COUNT_IF(customer_monetary_level IS NULL OR customer_monetary_level NOT IN ('Bassa spesa', 'Media spesa', 'Alta spesa', 'Cliente non disponibile')) AS failed_rows, COUNT(*) AS total_rows FROM payments.default.pay_transactions_features",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-046", "F019", "customer_monetary_level", "payments.default.pay_transactions_features", "Consistency",
 "Each client has exactly one tier",
 "SELECT COUNT_IF(n_levels > 1) AS failed_rows, COUNT(*) AS total_rows FROM (SELECT client_id, COUNT(DISTINCT customer_monetary_level) AS n_levels FROM payments.default.pay_transactions_features WHERE client_id IS NOT NULL GROUP BY client_id)",
 0, "Blocker", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active"),
 
("DQ-047", "F019", "customer_monetary_level", "payments.default.pay_transactions_features", "Accuracy",
 "Tier balance: each tier holds 20-45% of clients (tertiles)",
 "SELECT COUNT_IF((lvl = 'Bassa spesa' AND (share < 0.20 OR share > 0.45)) OR (lvl = 'Media spesa' AND (share < 0.20 OR share > 0.45)) OR (lvl = 'Alta spesa' AND (share < 0.20 OR share > 0.45))) AS failed_rows, COUNT(*) AS total_rows FROM (SELECT customer_monetary_level AS lvl, COUNT(DISTINCT client_id) / SUM(COUNT(DISTINCT client_id)) OVER () AS share FROM payments.default.pay_transactions_features WHERE client_id IS NOT NULL AND customer_monetary_level IN ('Bassa spesa', 'Media spesa', 'Alta spesa') GROUP BY customer_monetary_level)",
 0, "Warning", "After each run of feature_engineering.sql", "TO_DEFINE (data steward)", "Active")
AS v(check_id, feature_id, feature_name, target_table, dq_dimension, check_description, check_sql,
     max_failed_pct, severity, frequency, check_owner, status);
 
 
-- =====================================================================
-- TABLE 5: DATA QUALITY RESULTS (history)
-- IF NOT EXISTS on purpose: this is the audit trail of every check run,
-- so it must never be replaced. Filled by run_dq_checks.py.
-- =====================================================================
CREATE TABLE IF NOT EXISTS payments.default.dq_check_results (
  run_id         STRING    COMMENT 'One id per execution of the check suite',
  check_id       STRING    COMMENT 'dq_check_register.check_id',
  run_at         TIMESTAMP COMMENT 'When the check ran',
  failed_rows    BIGINT    COMMENT 'Rows failing the check',
  total_rows     BIGINT    COMMENT 'Rows evaluated',
  failed_pct     DOUBLE    COMMENT 'failed_rows / total_rows * 100',
  max_failed_pct DOUBLE    COMMENT 'Tolerance at run time',
  severity       STRING    COMMENT 'Severity at run time',
  result         STRING    COMMENT 'PASS / FAIL / ERROR',
  error_message  STRING    COMMENT 'Error text when result = ERROR'
)
COMMENT 'History of data quality check results';